In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import re
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)  # Show all columns in DataFrame display
import sys; sys.path.append('..')
from config import Paths
from eals_targetals.utils import make_article_table
from dataframes import *

In [28]:
df_demo = load_demographics_data()

### RAPA pALS vs Radcliff pALS

In [4]:
import sys; sys.path.append('/home/ubuntu/eals/eals_radcliff_feli/')
from eals_radcliff.utils import dataframes

In [5]:
# Load radcliff demographics data
df_rad_demo = dataframes.load_demographics_data()

# add year_of_diagnosis and symptom_onset_year
df_rad_demo['symptom_onset_year'] = df_rad_demo.symptom_onset_date.dt.year
s = df_rad_demo['date_of_diagnosis'].astype(str).str.strip()
s = s.str.replace(r'(?i)^O(?=\d)', '0', regex=True) # Replace leading/embedded 'O' that should be a zero ->  O7/15/2023 -> 07/15/2023
s = s.str.replace(r'(?i)(?<=\d)O(?=\d)', '0', regex=True) # 1O/15/2023 -> 10/15/2023 (if it happens)
df_rad_demo['year_of_diagnosis'] = pd.to_datetime(s, format='%m/%d/%Y', errors='coerce').dt.year

# Load radcliff ALSFRSR data to get age at enrollment
df_rad_als = dataframes.load_alsfrsr_data()
df_rad_als.date = pd.to_datetime(df_rad_als.date)
df_rad_als.sort_values(['user_id','date'], inplace=True)
df_rad_als = df_rad_als.groupby('user_id').first().reset_index()

# Add age at enrollment
age_at_enrollment = []
for idx, row in df_rad_demo.iterrows():
    pal_als = df_rad_als.query('user_id == @row.user_id')
    if pal_als.shape[0] == 0:
        r = np.nan
    else:
        r = pal_als.date.iloc[0].year - row['date_of_birth'].year
    age_at_enrollment.append(r)
df_rad_demo["age_at_enrollment"] = age_at_enrollment

In [6]:
# Merge Radcliff and RAPA data
df_rad = pd.merge(df_rad_demo, df_rad_als, on='user_id', how='outer')
df_rad['study'] = 'Radcliff'
print(df_rad.shape)

df_rapa_als = load_alsfrsr_data()
df_rapa_als.date = pd.to_datetime(df_rapa_als.date)
df_rapa_als.sort_values(['user_id','date'], inplace=True)
df_rapa_als = df_rapa_als.groupby('user_id').first().reset_index()
df_rapa = df_demo.merge(df_rapa_als, on='user_id', how='outer')
df_rapa['study'] = 'RAPA'
print(df_rapa.shape)

df_comparison = pd.concat([df_rad, df_rapa], axis=0, ignore_index=True, sort=False)
print(df_comparison.shape)

(156, 97)
(22, 101)
(178, 107)


In [7]:
# Example usage
cat_columns = [
    'birth_gender',
    'symptom_onset_site',
    'specific_limb_onset', 
    'family_neuro_history',
    'muscle_disease',
    'has_fallen',
    'any_walking_aid',
    'anxiety', 
    'depression',
]
numeric_columns = [
    'age',
    'height_in_m',
    'bmi',
    'years_since_onset',
    'ALS_total',
    'speech',
    'salvation',
    'swallowing',
    'handwriting',
    'cutting_food_a',
    'cutting_food_b',
    'dressing_and_hygiene',
    'turning_in_bed',
    'walking',
    'climbing_stairs',
    'dyspnea',
    'orthopnea',
    'respiratory_insufficiency',
    'bulbar_subscore',
    'respiratory_subscore',
    'fine_motor_subscore',
    'gross_motor_subscore',

]

In [8]:
# This columns are not being compared in the table below
dis = []
for c in df_rapa.columns:
    if c not in cat_columns and c not in numeric_columns:
        dis.append(c)
df_rapa[dis].head(2)

,user_id,family_disease_details,adhd_add,als,bipolar_disorder,brain_aneurysm,Bulbar,c9orf_mutation,cancer,residence_location,birth_location,clinic_city,clinic_name,cognitive_impairment,concussions,Consent form signed Date,consent_signed,date_of_birth,date_of_diagnosis,symptom_onset_date,dementia,diabetes,diagnosis,Do You Use The Railing When Climbing Stairs?,additional_conditions,current_cold,stairs_at_home,Do you use railing when climbing stairs?,epilepsy,ethnicity,hearing_loss,recent_hospitalization,speech_therapy,recent_quit_smoking,height_feet,height_inch,education_level,stairs_description,number_of_steps,english_first_language,symptom_onset_site_limb,marital_status,No,specific_allergies,step_height,specified_dimensions,race,residence_type,residence_type_2,schizophrenia,smoking_history,household_income,weight,current_prescriptions,stairs_flooring,dominant_hand,occupation,orthotics_type,study_shoes_type,us_region_identity,neurologist_physician_name,other_education,other language,other_symptoms,date_x,weight_in_kg,session_id_x,created_at,date_y,session_id_y,months_since_first_session,years_since_first_session,study
0,1b0b349e-301f-46a4-a137-1fed70ca678d,None,No,Known ALS Gene,No,No,Swallowing,No,No,"Tustin, California",Mexico,Orange,UCI,No,No,None,Yes,1979-07-03,12/05/2024,2023-08-01,No,No,ALS,None,None of the above,No,None,None,No,"Mexican, Mexican American, Chicano/a",No,No,"yes, not currently",No,5.0,5.0,High School,None,None,No,None,Single,Spanish,[],None,None,White,None,Condo/townhouse,No,I have never smoked,None,148.0,[],None,Right,"Disability, Not working at this time",[],None,West,Dr. Jeffery Mullen,None,None,None,2025-05-22,67.131616,1b0b349e-301f-46a4-a137-1fed70ca678d__2025-05-22,2025-07-10 21:11:01+00:00,2025-07-10,1b0b349e-301f-46a4-a137-1fed70ca678d__2025-07-10,0.0,0.0,RAPA
1,1be5b39e-e8b8-47fc-9705-67849c71db69,None,No,None,No,No,Speech,No,No,"Chester, New Hampshire",Vietnam,"Boston, MA",MGH,No,No,None,None,1974-12-23,03/01/2023,2021-11-01,No,Yes,ALS,None,None of the above,"No, 10/21/2024",None,None,No,"Not of Hispanic, Latino/a or Spanish origin",No,"No, 10/21/2024",No,No,5.0,2.0,Vocational Training,None,None,Yes,None,Married,None,"[""Pollen"",""Dogs""]",None,None,Chinese,None,House,No,I have never smoked,None,130.0,"[""Claritin, PRN""]",None,Right,Account Manager,[],-,Northeast,Dr. Garret,None,None,None,2024-10-21,58.966960,1be5b39e-e8b8-47fc-9705-67849c71db69__2024-10-21,NaT,NaT,NaN,NaN,NaN,RAPA


In [9]:
styled_table = make_article_table(df_comparison, 'study', [], numeric_columns, dec=2, p_dec=2)
display(styled_table)

,,RAPA,Radcliff,p-value (T-Test/Chi-squared)
Variable,Value,,,
Age,Average (std) [N],57.80 (13.38) [20],59.59 (11.24) [141],0.57
Height In M,Average (std) [N],2.36 (2.93) [20],1.72 (0.09) [142],0.34
Bmi,Average (std) [N],23.23 (7.10) [20],27.24 (4.84) [142],0.023
Years Since Onset,Average (std) [N],5.40 (9.66) [19],4.25 (4.70) [132],0.62
Als Total,Average (std) [N],24.06 (8.91) [17],40.32 (5.53) [143],9.3e-07
Speech,Average (std) [N],2.41 (1.62) [17],3.50 (0.63) [143],0.014
Salvation,Average (std) [N],3.00 (1.12) [17],3.57 (0.74) [143],0.054
Swallowing,Average (std) [N],3.00 (1.27) [17],3.62 (0.57) [143],0.063
Handwriting,Average (std) [N],1.71 (1.53) [17],3.43 (0.60) [143],0.00028


In [10]:
styled_table = make_article_table(df_comparison, 'study', cat_columns, [], dec=2, p_dec=2)
display(styled_table)

Exact FFH not available. Falling back to chi-square p-value. Install 'fisher' for exact.


In [20]:
available = [c for c in cat_columns if c in df_comparison.columns]
df_comparison.query('study=="RAPA"')[available].info()

<class 'pandas.core.frame.DataFrame'>
Index: 22 entries, 156 to 177
Data columns (total 6 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   birth_gender          20 non-null     object
 1   symptom_onset_site    20 non-null     object
 2   family_neuro_history  19 non-null     object
 3   muscle_disease        20 non-null     object
 4   anxiety               20 non-null     object
 5   depression            20 non-null     object
dtypes: object(6)
memory usage: 1.2+ KB


In [27]:
sus = ['b45d5623-dc50-4529-bf2c-dc45bc22c0b4', 'b96c8a5b-8902-423a-8ced-18f6b01e9885']
df_comparison.query('user_id in @sus')

,user_id,family_disease_details,adhd_add,als,anxiety,bipolar_disorder,brain_aneurysm,Bulbar,c9orf_mutation,cancer,residence_location,birth_location,clinic_city,clinic_name,cognitive_impairment,concussions,consent_signed,Consent form signed?--Feb 2024,date_of_birth,date_of_diagnosis,symptom_onset_date,dementia,depression,diabetes,diagnosis,additional_conditions,current_cold,family_neuro_history,stairs_at_home,epilepsy,ethnicity,hearing_loss,recent_hospitalization,speech_therapy,recent_quit_smoking,height_feet,height_inch,education_level,stairs_description,number_of_steps,english_first_language,symptom_onset_site_limb,marital_status,muscle_disease,birth_gender,specific_allergies,step_height,specified_dimensions,race,residence_type_2,schizophrenia,symptom_onset_site,smoking_history,household_income,weight,current_prescriptions,stairs_flooring,dominant_hand,occupation,orthotics_type,study_shoes_type,us_region_identity,neurologist_physician_name,other_education,other_race,other_symptoms,date_x,height_in_m,weight_in_kg,bmi,age,years_since_onset,symptom_onset_year,year_of_diagnosis,age_at_enrollment,speech,salvation,swallowing,handwriting,cutting_food_a,cutting_food_b,dressing_and_hygiene,turning_in_bed,walking,climbing_stairs,dyspnea,orthopnea,respiratory_insufficiency,ALS_total,created_at,date_y,bulbar_subscore,respiratory_subscore,fine_motor_subscore,gross_motor_subscore,session_id,study,Consent form signed Date,Do You Use The Railing When Climbing Stairs?,Do you use railing when climbing stairs?,No,residence_type,other language,session_id_x,session_id_y,months_since_first_session,years_since_first_session
170,b45d5623-dc50-4529-bf2c-dc45bc22c0b4,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,Yes,NaN,NaT,None,NaT,None,None,None,None,None,None,None,None,None,None,None,None,None,None,NaN,NaN,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,NaN,None,None,None,None,None,None,None,None,None,NaN,None,2026-01-19,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT,NaN,NaN,NaN,NaN,NaN,RAPA,2026-01-19,None,None,None,None,None,b45d5623-dc50-4529-bf2c-dc45bc22c0b4__2026-01-19,NaN,NaN,NaN
171,b96c8a5b-8902-423a-8ced-18f6b01e9885,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,Yes,NaN,NaT,None,NaT,None,None,None,None,None,None,None,None,None,None,None,None,None,None,NaN,NaN,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,NaN,None,None,None,None,None,None,None,None,None,NaN,None,2025-11-14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0,3.0,3.0,3.0,2.0,NaN,2.0,2.0,2.0,0.0,1.0,1.0,4.0,26.0,2025-11-14 15:16:41+00:00,2025-11-14,9.0,6.0,NaN,4.0,NaN,RAPA,2025-11-14,None,None,None,None,None,b96c8a5b-8902-423a-8ced-18f6b01e9885__2025-11-14,b96c8a5b-8902-423a-8ced-18f6b01e9885__2025-11-14,0.0,0.0


In [25]:
df_comparison.query('study=="RAPA"')#]

,user_id,family_disease_details,adhd_add,als,anxiety,bipolar_disorder,brain_aneurysm,Bulbar,c9orf_mutation,cancer,residence_location,birth_location,clinic_city,clinic_name,cognitive_impairment,concussions,consent_signed,Consent form signed?--Feb 2024,date_of_birth,date_of_diagnosis,symptom_onset_date,dementia,depression,diabetes,diagnosis,additional_conditions,current_cold,family_neuro_history,stairs_at_home,epilepsy,ethnicity,hearing_loss,recent_hospitalization,speech_therapy,recent_quit_smoking,height_feet,height_inch,education_level,stairs_description,number_of_steps,english_first_language,symptom_onset_site_limb,marital_status,muscle_disease,birth_gender,specific_allergies,step_height,specified_dimensions,race,residence_type_2,schizophrenia,symptom_onset_site,smoking_history,household_income,weight,current_prescriptions,stairs_flooring,dominant_hand,occupation,orthotics_type,study_shoes_type,us_region_identity,neurologist_physician_name,other_education,other_race,other_symptoms,date_x,height_in_m,weight_in_kg,bmi,age,years_since_onset,symptom_onset_year,year_of_diagnosis,age_at_enrollment,speech,salvation,swallowing,handwriting,cutting_food_a,cutting_food_b,dressing_and_hygiene,turning_in_bed,walking,climbing_stairs,dyspnea,orthopnea,respiratory_insufficiency,ALS_total,created_at,date_y,bulbar_subscore,respiratory_subscore,fine_motor_subscore,gross_motor_subscore,session_id,study,Consent form signed Date,Do You Use The Railing When Climbing Stairs?,Do you use railing when climbing stairs?,No,residence_type,other language,session_id_x,session_id_y,months_since_first_session,years_since_first_session
156,1b0b349e-301f-46a4-a137-1fed70ca678d,None,No,Known ALS Gene,No,No,No,Swallowing,No,No,"Tustin, California",Mexico,Orange,UCI,No,No,Yes,NaN,1979-07-03,12/05/2024,2023-08-01,No,No,No,ALS,None of the above,No,None,None,No,"Mexican, Mexican American, Chicano/a",No,No,"yes, not currently",No,5.0,5.0,High School,None,None,No,None,Single,No,Female,[],None,None,White,Condo/townhouse,No,Bulbar,I have never smoked,None,148.0,[],None,Right,"Disability, Not working at this time",[],None,West,Dr. Jeffery Mullen,None,NaN,None,2025-05-22,1.6510,67.131616,24.628216,46.0,1.806982,NaN,NaN,NaN,2.0,2.0,3.0,3.0,1.0,2.0,2.0,3.0,3.0,3.0,3.0,2.0,2.0,29.0,2025-07-10 21:11:01+00:00,2025-07-10,7.0,7.0,8.0,9.0,NaN,RAPA,None,None,None,Spanish,None,None,1b0b349e-301f-46a4-a137-1fed70ca678d__2025-05-22,1b0b349e-301f-46a4-a137-1fed70ca678d__2025-07-10,0.0,0.0
157,1be5b39e-e8b8-47fc-9705-67849c71db69,None,No,None,Yes,No,No,Speech,No,No,"Chester, New Hampshire",Vietnam,"Boston, MA",MGH,No,No,None,NaN,1974-12-23,03/01/2023,2021-11-01,No,Yes,Yes,ALS,None of the above,"No, 10/21/2024",No,None,No,"Not of Hispanic, Latino/a or Spanish origin",No,"No, 10/21/2024",No,No,5.0,2.0,Vocational Training,None,None,Yes,None,Married,No,Female,"[""Pollen"",""Dogs""]",None,None,Chinese,House,No,Bulbar,I have never smoked,None,130.0,"[""Claritin, PRN""]",None,Right,Account Manager,[],-,Northeast,Dr. Garret,None,NaN,None,2024-10-21,1.5748,58.966960,23.777048,50.0,2.970568,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT,NaN,NaN,NaN,NaN,NaN,RAPA,None,None,None,None,None,None,1be5b39e-e8b8-47fc-9705-67849c71db69__2024-10-21,NaN,NaN,NaN
158,623d5c97-db0b-41a5-aaac-9fd096ca3fa5,"Mother, 70, Dementia",Yes,Unsure,No,No,No,None,No,No,"Beaverton, OR","Hillsboro, OR","Providence, OR",Providence Neurological Eest,No,Yes,Yes,NaN,1966-07-21,04/24/2025,2024-10-01,No,Yes,Yes,ALS,Vision impaired / wear glasses,No,Yes,No,No,"Not of Hispanic, Latino/a or Spanish origin",No,No,No,No,5.0,8.0,College,None,None,Yes,None,Single,No,Male,"[""Pollen, Grass""]",None,None,White,Condo/townhouse,No,Respiratory,I have never smoked,60000,178.0,[],None,Right,"Disability 2025, in I.T",[],None,West,Dt. Nicholas Olney,None,NaN,None,2025-11-11,1.7272,80.739376,27.064508,59.0,1.111567,NaN,NaN,NaN,3.0,3.0,4.0,2.0,3.0,3.0,4.0,4.0,4.0,2.0,3.0,2.0,2.0,36.0,2025-12-15 18:56:33+00:00,2025-12-1